### Create Hakai work-funder linkages (OUTPUT-LIST pattern)

Pilot of the **OUTPUT-LIST pattern** (how-to-add-a-funder-v2 **§11**). Resolves the
DOIs Hakai publishes on its own funded-publications list (scraped by
`scripts/local/hakai_outputs_to_s3.py` from https://hakai.org/publications) to OpenAlex
works, and writes the junction `openalex.awards.hakai_work_funders (work_id, funder_id,
provenance)`.

**No award entities, no grants, no `/awards` rows.** This is a first-party funder→work
assertion that lands **directly in `work.funders`** via the `from_funder_reported` leg of
`notebooks/end2end/CreateWorksEnriched.ipynb` (mirror of the existing `from_crossref`
leg). Funder-asserted edges are trusted above Crossref/text-mined edges, so no per-edge
provenance is attached downstream (provenance is kept here for auditing only).

**Funder:** Hakai Institute `F4320334031` (parent Tula Foundation `F4320315065`). The
list is published by Hakai, so the edge attaches to Hakai.

**DOI cleaning is done in the scraper (Python)** — the parquet `doi` column is already a
canonical lowercase `https://doi.org/…` string, so the join below is plain (no SQL regex,
no escapedStringLiterals backslash hazard).

**Deploy order (§11.4 / §11.6):** run this notebook FIRST so the junction + shared
`funder_reported_work_funders` table exist, THEN add the `from_funder_reported` leg to
CreateWorksEnriched and run `walden_end2end`. Adding the leg before the table exists
fails the next run with table-not-found.

#### Step 1 — stage the scraped outputs parquet

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hakai_outputs_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hakai/hakai_outputs.parquet`;

In [ ]:
%sql
-- Inspect before transforming (§2 convention): schema + a sample + the row count.
DESCRIBE openalex.awards.hakai_outputs_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hakai_outputs_raw LIMIT 10;

#### Step 2 — resolve DOIs to works and build the junction

Path 1 (DOI) only — Hakai publishes clean DOIs. DOIs not indexed in OpenAlex simply fail
to join and drop out (coverage is never 100%; that is expected, not a failure).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hakai_work_funders
USING delta AS
WITH outputs AS (
  SELECT DISTINCT
    lower(doi)            AS doi_url,
    CAST(funder_id AS BIGINT) AS funder_id,
    provenance
  FROM openalex.awards.hakai_outputs_raw
  WHERE doi IS NOT NULL AND doi <> ''
),
resolved AS (
  SELECT DISTINCT w.id AS work_id, o.funder_id, o.provenance
  FROM outputs o
  JOIN openalex.works.openalex_works w
    ON lower(w.doi) = o.doi_url
)
SELECT work_id, funder_id, MAX(provenance) AS provenance
FROM resolved
GROUP BY work_id, funder_id;

#### Step 3 — (re)build the shared `funder_reported_work_funders` table

The `from_funder_reported` leg in CreateWorksEnriched reads this ONE table, so the leg
never needs editing as funders are added — just `UNION ALL` the next junction in here.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.funder_reported_work_funders
USING delta AS
SELECT work_id, funder_id, provenance FROM openalex.awards.hakai_work_funders
UNION ALL SELECT work_id, funder_id, provenance FROM openalex.awards.europepmc_work_funders   -- oxjob #477 EuropePMC
-- UNION ALL SELECT work_id, funder_id, provenance FROM openalex.awards.sfari_work_funders   -- add when SFARI lands
-- UNION ALL SELECT work_id, funder_id, provenance FROM openalex.awards.dndi_work_funders    -- add when DNDi lands
;

#### Sanity checks

In [ ]:
%sql
-- Coverage = junction_rows / scraped_dois (under 100% is normal: some DOIs aren't in OA).
-- distinct_keys must equal junction_rows (no duplicate (work_id, funder_id)).
SELECT
  (SELECT COUNT(*) FROM openalex.awards.hakai_outputs_raw)                                   AS scraped_dois,
  (SELECT COUNT(*) FROM openalex.awards.hakai_work_funders)                                  AS junction_rows,
  (SELECT COUNT(DISTINCT concat(work_id, ':', funder_id)) FROM openalex.awards.hakai_work_funders) AS distinct_keys,
  (SELECT COUNT(DISTINCT work_id) FROM openalex.awards.hakai_work_funders)                   AS distinct_works;

In [ ]:
%sql
-- Funder must exist in the dim (F4320334031 = Hakai Institute). Expect exactly 1 row.
SELECT funder_id, display_name, ror_id FROM openalex.mid.funder WHERE funder_id = 4320334031;

#### Make it live (§11.4 / §11.6)

1. Add the `from_funder_reported` leg to the **"Merge funders"** cell of
   `notebooks/end2end/CreateWorksEnriched.ipynb` and UNION it into `unioned` (one-time):

   ```sql
   from_funder_reported AS (
     SELECT
       fr.work_id,
       CONCAT("https://openalex.org/F", fr.funder_id) AS funder_id,
       f.ror_id       AS ror,
       f.display_name AS display_name
     FROM openalex.awards.funder_reported_work_funders fr
     JOIN openalex.mid.funder f ON f.funder_id = fr.funder_id
   )
   ```

2. Run `walden_end2end` (CreateWorksEnriched rebuilds `work.funders`, then works→ES).
   The `/awards` sync does NOT update works — this leg is what surfaces it on `/works`.
3. Confirm:
   `curl -s 'https://api.openalex.org/works?filter=funder.id:F4320334031&per_page=1' | jq '.meta.count'`
   — should be non-zero and in the same ballpark as `distinct_works` above.